# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型  |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** どこから読み、最初に何をするか |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# コンテナの更新と追加

**当てるのは人。機械は知らせるだけ。** 上げる前に必ず控えを取る。上げたら**画面で確かめる**。
対象は本番だけ(検証機は 2026-09-07 に退役)。**試す場所は無い**という前提で読む。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 1. いまの形

```
プロジェクト  kosenmap        ← 2026-09-07 に Test から改名
コンテナ      km-*            ← 同上。dev-* から改名
ボリューム    test_*          ← 実体名は据え置き(compose の volumes: の name: で固定)
```

`.env` に `COMPOSE_FILE="compose.yaml:compose.vps.yaml"` が入っているので、**ホストでは素の `docker compose` で両方が読まれる。**

| サービス | image | 固定の仕方 |
|---|---|---|
| `logto` / `postgres` / `mariadb` / `phpmyadmin` | `名前:タグ@sha256:…`(例 `ghcr.io/logto-io/logto:1.43.0@sha256:…`) | **タグと digest の両方**(2026-09-15)。動くのは digest。タグは `check-updates.sh` が版番号と「固定より新しい修正版」を読むために残す |
| `reverse-proxy` / `mailpit` / `certbot` / `mailserver` | `…@sha256:…` | **digest** |
| `web` / `soketi` | `build:`(自前) | Dockerfile の `FROM` を `タグ@digest` で固定。ビルド元の更新は `base` として知らせる。**compose に digest を貼らない** |

**`:latest` は1つも無い**(`check.php` の `compose` が見張る)。

**本番では `mailpit` を起動しない**(`compose.vps.yaml` の `profiles`)。上限・利用者の決まりは §1-1。

### いまの像

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker ps --format '{{.Names}}\t{{.Image}}\t{{.Status}}' | sort

### 1-1. 上限・起動しないもの・利用者(2026-09-14 の診断で追加)

**コンテナごとの上限。** それまでどのサービスにも上限が無く、1つの暴走(PHP のメモリ・Soketi の接続・fork の連鎖)が**ホスト全体(RAM 2GB)を道連れにできた。**
本番の実測の 2〜4 倍を目安に置いている。全員が同時に使い切る前提の枠ではなく、**1つが暴れても他を巻き込まない**ための天井。

| サービス | `mem_limit` | `pids_limit` | `no-new-privileges` |
|---|---|---|---|
| `logto` | 512m | 256 | あり |
| `mariadb` | 512m | 200 | あり |
| `web` | 384m | 200 | あり |
| `postgres` / `soketi` / `phpmyadmin` | 256m | 200 | あり |
| `reverse-proxy` / `mailpit` / `certbot` | 128m | 200 | あり |
| `mailserver` | 192m | 200 | **付けない**(postfix の `postdrop` が setgid で動くので、付けると送信が止まる) |

`certbot` と `mailserver` の分は `compose.vps.yaml`、残りは `compose.yaml`(VPS にもそのまま効く)。
`no-new-privileges` は setuid で権限を上げる道を塞ぐだけで、root で起動してから一般利用者へ下りる形(Apache・nginx・gosu)は妨げない。

**`web` は Apache のワーカーが最大 150 のまま**なので、重い PHP が並ぶと 384m に当たりうる。配備後の数日は下のセルで `OOMKilled` と使用量を見る。
当たるなら `docker/php/apache-km.conf` で `MaxRequestWorkers` を絞るか、`mem_limit` を上げる。`mem=0` / `pids=0` は上限なし(配備前の形)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker inspect --format '{{.Name}}  mem={{.HostConfig.Memory}}  pids={{.HostConfig.PidsLimit}}  {{.HostConfig.SecurityOpt}}  OOMKilled={{.State.OOMKilled}}' $(docker compose ps -q) | sed 's#^/##' | sort
echo
docker stats --no-stream --format '{{.Name}}\t{{.MemUsage}}\t{{.PIDs}}' | sort

**本番では Mailpit を起動しない。** 本番の送信は `mailserver` が担い、Mailpit は1通も受けていなかった(実測 0 通)のに、無認証の画面を持つサービスが devnet に居続けていた。
`compose.vps.yaml` で `profiles: ["mailpit"]` を付けたので、`docker compose up -d` では作られない(`reverse-proxy` も Mailpit を待たない)。

- nginx の 8025 の受け口は Mailpit が居なくても起動する。**本番では 2026-09-18 から publish しないので、外からは繋がらない**(管理画面のサービス一覧のリンクも本番では開かない)
- 検証で一時的に使うなら `docker compose --profile mailpit up -d mailpit`。済んだら下のセルで片付ける
- **既に動いているコンテナは `up` しても消えない。** 配備のあとに一度だけ下のセルを流す(メールは 0 通なので失うものは無い)
- 校内 LAN の構成(`compose.yaml` 単体)では従来どおり起動する

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Mailpit のコンテナを止めて消します(本番では使っていません。メールは 0 通)"
docker compose --profile mailpit stop mailpit && docker compose --profile mailpit rm -f mailpit
docker ps --format '{{.Names}}' | grep -x km-mailpit || echo "km-mailpit は居ません"

**Soketi は `node` 利用者で動かす**(`docker/soketi/Dockerfile` の `USER node`)。それまでは root で動いていた。
待ち受けは 6001 / 9601 で特権は要らず、書き込むのは `PM2_HOME` の `/tmp` だけ。
**イメージを作り直さないと効かない。** 配備のあとに `docker compose build soketi` → `docker compose up -d`(§5-4 のセルでもよい)。下のセルで `root` と出たら、まだ作り直していない。

**Node 16 はやめた**(2026-09-15)。上流の soketi 1.6.1 は Node 14・16・18 でしか起動しないので、Node 22/24 に載せ替えたフォーク
aloware/soketi の `/app` を、Debian 13 の `node:24-trixie-slim` に載せて使う(`docker/soketi/Dockerfile`。**フォークが配るイメージは GLIBC が足りず起動しない**)。
通信・HTTP API・環境変数は 1.6.1 と同じ。経緯と確かめたことは [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) §2。下のセルで `v24` と出れば入れ替わっている。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose exec -T soketi sh -c 'echo "利用者: $(id -un)(uid $(id -u)) / Node: $(node -v)"'

**phpMyAdmin に root で入れない**(`docker/phpmyadmin/config.user.inc.php`)。イメージの既定は root でも空のパスワードでも入れた(本番の 5.2.3 で実測)。
いまは `AllowRoot` を `.env` の `KM_PMA_ALLOW_ROOT` が `1` のときだけ許し、`AllowNoPassword` は常に禁止、操作が無いまま 30 分でログインを切る。

普段の作業はアプリの利用者(`MARIADB_USER`)で足りる。どうしても root が要るときだけ:

1. `.env` に `KM_PMA_ALLOW_ROOT=1` を書き、`docker compose up -d`(変わった phpmyadmin だけが作り直される)
2. 作業する
3. **済んだら行を空に戻して、もう一度 `docker compose up -d`**

`1` 以外(空・`0`・`true` など)はすべて禁止のまま。配備前は設定ファイルが無いので、この値に関係なく root で入れる。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
if docker compose exec -T phpmyadmin test -f /etc/phpmyadmin/config.user.inc.php </dev/null; then
  echo "config.user.inc.php: あり"
else
  echo "config.user.inc.php: ★ ありません(配備前の形。root で入れます)"
fi
echo "KM_PMA_ALLOW_ROOT: '$(docker compose exec -T phpmyadmin printenv KM_PMA_ALLOW_ROOT </dev/null 2>/dev/null | tr -d '\r')'(1 のときだけ root を許す)"

## 2. なぜ自動で当てないのか

| | 自動でよいか | なぜ |
|---|---|---|
| Ubuntu のセキュリティ更新 | **よい**(unattended-upgrades) | ただし再起動は自動にしない |
| **Logto** | **駄目** | 起動時に DB の移行が走る。**Logto は管理画面のゲートそのもの**なので、落ちると直しに行く手段ごと失う |
| MariaDB / PostgreSQL | **駄目** | データの入れ物。戻せない |
| nginx / certbot / postfix / mailpit | 人が見てから | 入口・証明書・メールが止まる |

## 3. 版を固定する(`--pins`)

ホストで出した `image:` 行を `compose.yaml` / `compose.vps.yaml` へ貼る。**`web` / `soketi` / `logto` は貼らない**(出てこない)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
./scripts/check-updates.sh --path /opt/kosenmap --pins

## 4. 更新を見つける

毎日 4:17 に調べて、変化があればメールが来る。手で見るならこのセル。

| 種類 | 何が分かるか | 打つ手 |
|---|---|---|
| `release` | 新しい版が公開された(Logto) | 上げるか判断する(§5-2) |
| `digest` | 動くタグの中身が入れ替わった | 気づかないうちに上がりうる。固定する |
| `series` | 系列のタグ(`11.4` など)の中身が入れ替わった = **修正版** | 控え → `pull` → `up -d`(§5-3) |
| `base` | 自前ビルドのビルド元が入れ替わった | 控え → `build --pull` → `up -d`(§5-4)。**pull では当たらない** |

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
./scripts/check-updates.sh --path /opt/kosenmap

## 5. 当て方

### 5-1. 共通の手順(この順番から外れない)

1. **控えを取る**([03-backup](03-backup.ipynb) の「いま控えを取る」)。`summary.txt` の行数を見る
2. 手元の compose を書き換える。**版は必ず具体的な値**(`latest` に戻さない)
3. `check.php` を通す([01-daily-check](01-daily-check.ipynb))
4. 配備する。**compose を変えたら `-Action up`**([02-deploy](02-deploy.ipynb))
5. **§7 の一覧を画面で確かめる**

### 5-3. 修正版(series)を当てる

compose は書き換えない(タグは同じ)。**先に控えを取ってから。** `SERVICES` を当てるものに書き換える。
**メジャー版は上げない**(`postgres:17` → `18` はデータ領域の形式が変わり、そのままでは起動しない)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "指定したサービスの像を取り直して入れ替えます(先に控えを取りましたか?)" --timeout 900
SERVICES="mariadb phpmyadmin"
docker compose pull $SERVICES
docker compose up -d
docker compose ps

### 5-4. ビルド元(base)を当てる

`web` / `soketi` は自前でビルドする。**pull では変わらない。** `src/` は bind mount なので、PHP のコードだけならビルドは要らない。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "web と soketi をビルドし直して入れ替えます(先に控えを取りましたか?)" --timeout 1800
docker compose build --pull web soketi
docker compose up -d
docker compose ps

### 5-2. Logto(一段重い)

- **1つずつ上げる**(飛ばさない)。**会期中に上げない**。リリースノートの breaking change を読む
- 上げた直後に**サインインを実際に通す**(管理画面と Android の両方)
- **`docker compose up -d logto` だけを叩かない**(`depends_on` が評価されない)。全体の `up -d` にする

手順:

1. **控えを取る**(飛ばすと戻れない)
2. 手元の `compose.yaml` の logto を新しい版にする(digest にしない)
3. `check.php` → **`-Action deploy`**(`up` にしない。先に移行を当てるので、ここではファイルを置くだけ)
4. 新しい像を取る → 5. 移行を**新しい像で**当てる → 6. 立て直す → 7. 起動を見届ける → 8. §7 を確かめる

**`NEW` を上げる先の版に書き換えてから**、4 から順に実行する。

### 4. 新しい像を取る

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto の新しい像を取ります(まだ入れ替えません)" --timeout 900
docker compose pull logto

### 5a. 移行の中身を見る

使い捨てのコンテナを新しい像で立てて、**当てずに一覧だけ**出す。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --confirm "使い捨ての Logto コンテナを立てて、移行の一覧を出します(DB は書き換えません)" --timeout 900
NEW=1.44.0
docker compose run --rm --entrypoint sh logto -c "cd /etc/logto && npm run alteration list $NEW"

### 5b. 移行を当てる

動いている古いコンテナで叩いても、新しい移行は入っていないので何も起きない。**新しい像で**走らせる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto の DB 移行を当てます。控えは取りましたか?(戻すのは難しい)" --timeout 1800
NEW=1.44.0
docker compose run --rm --entrypoint sh logto -c "cd /etc/logto && npm run alteration deploy $NEW"

### 6. 立て直す

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "docker compose up -d で Logto を新しい版に入れ替えます" --timeout 900
docker compose up -d

### 7. 起動を見届ける

`migration` の行が並んで終わっていれば通っている。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose logs --tail 80 logto

### 5-5. nginx の設定が読めているか

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker exec km-nginx-proxy nginx -t

## 6. 戻し方

| 状況 | 戻し方 |
|---|---|
| DB に触らない像(nginx / certbot / postfix / mailpit)を上げた | compose の版を戻して `-Action up` |
| **Logto を上げて移行が走った** | **タグを戻すだけでは直らない。** 巻き戻し(下)か、控えからの復元 |
| MariaDB / PostgreSQL を上げた | 控えからの復元([03-backup](03-backup.ipynb) §8) |

**巻き戻しは万能ではない。** 列や表を落とす移行を戻しても、落ちた中身は返ってこない。確実なのは復元。
`PREV` を戻す先の版にしてから実行する。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto の DB 移行を巻き戻します(控えは取りましたか?)" --timeout 1800
PREV=1.43.0
docker exec km-logto sh -c "cd /etc/logto && npm run alteration rollback $PREV"

## 7. 上げたあとに見るもの

**healthy だけでは足りない**(中の HTTP が返ることしか言っていない)。

- [ ] `https://ito4.jp/` が開く
- [ ] 地図が出る。**教職員氏名の解除が効く**
- [ ] **管理画面にサインインできる**(= Logto が生きている)
- [ ] Android アプリでサインインできる
- [ ] 管理画面のチャットと死活監視が「接続済み」になる(Soketi を作り直したとき)
- [ ] ファイル管理から**実際にダウンロードできる**
- [ ] 問い合わせフォームから1通送って、**メールが届く**(送信側からは成功に見える)

## 8. コンテナを足す

### 8-1. 先に決めること

| 問い | なぜ |
|---|---|
| 外に出すのか | 出すなら **nginx 越し**。`ports:` を直に足さない |
| データを持つのか | ボリュームに **`name:` を必ず書く** |
| 秘密を持つのか | `.env` に置く。compose に直書きしない。**秘密に既定値を書かない** |
| 落ちたら何が止まるのか | `depends_on` の条件を決める材料 |

### 8-2. compose に書く(最低限そろえるもの)

```yaml
  たとえば-redis:
    image: redis:7.4-alpine@sha256:… # latest にしない。タグ@digest で固定する(check.php が見張る)
    container_name: km-redis         # km- で始める
    restart: unless-stopped
    security_opt:
      - no-new-privileges:true       # setuid で権限を上げる道を塞ぐ(mailserver のような例外は §1-1)
    mem_limit: 256m                  # 上限を必ず書く。1つの暴走でホストを道連れにしない
    pids_limit: 200
    read_only: true                  # イメージの中に書かせない。書く場所は tmpfs とボリューム(docs/12 §2)
    tmpfs:
      - /tmp
    cap_drop:
      - ALL                          # 要る特権だけ cap_add で戻す(check.php が見張る)
    expose:
      - "6379"                       # ports: ではなく expose:
    volumes:
      - redis_data:/data
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 30s
      timeout: 5s
      retries: 3
    networks:
      - edge                         # nginx から届く網。DB を使うなら appdb も。忘れると誰からも見えない
```

末尾の `volumes:` に:

```yaml
  redis_data:
    name: kosenmap_redis_data        # name: を必ず書く(check.php が見張る)
```

**`name:` が無いと**、プロジェクト名を変えた瞬間に compose は「無いので空を新規作成」して**起動に成功したまま中身が空になる。**
compose に `build:` を足したら、`check-updates.sh` の `BUILD_DOCKERFILES` にも足す。

### 8-3. 外に出すなら nginx を通す

`nginx/default.conf.template` の冒頭(upstream の並び)に `upstream km_example { zone km_up_example 64k; server たとえば-redis:6379 resolve; }` を足し、`location` では `proxy_pass http://km_example;` と書く。`server` の名前は**サービス名**(コンテナ名ではない)。
**`proxy_pass http://たとえば-redis:6379;` と直に書かない。** nginx は起動時に引いた IP を持ち続けるので、相手のコンテナを作り直すと IP が変わって **502** になる(2026-09-14 に Logto で発生)。`resolve` を付けると動いている間も引き直す。`check.php` が直書きを見つけて落ちる。
管理系にするなら `include /etc/nginx/km/allow-*.conf;` と `include /etc/nginx/km/gate-location.conf;` を **`proxy_pass` の手前**に置く。

### 8-4. 配備して確かめる

`check.php` → `-Action up -Backup`([02-deploy](02-deploy.ipynb))→ 下のセル → §7。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose ps
echo
docker volume ls --format '{{.Name}}' | sort

## 9. やってはいけないこと

| | なぜ |
|---|---|
| `image:` を `latest` にする | 誰も決めないまま版が上がる |
| ボリュームの `name:` を省く | 改名した瞬間、起動に成功したまま中身が空になる |
| 新しいサービスに `ports:` を足す | TLS・IP 制限・ログイン判定を迂回する口が増える |
| nginx の `proxy_pass` にコンテナ名を書く | 改名で壊れる。サービス名で書く |
| nginx で `proxy_pass http://サービス名:ポート;` と直に書く | 起動時の IP を持ち続け、相手を作り直すと 502。`upstream … resolve` を通す |
| Logto を控え無しで上げる | 移行は戻せない |
| `docker compose up -d <1つだけ>` | `depends_on` が評価されない |
| **`docker compose down -v`** | **ボリュームごと消す。** DB も証明書も DKIM 鍵も消える |
| 新しいサービスに `mem_limit` / `pids_limit` を書かない | 1つの暴走でホスト全体(RAM 2GB)が止まる |
| `mailserver` に `no-new-privileges` を付ける | `postdrop` が setgid で動くので送信が止まる(送信側からは成功に見える) |
| `KM_PMA_ALLOW_ROOT=1` を置いたままにする | phpMyAdmin が MariaDB の root のパスワードを試す場所に戻る |
| healthy を見て終わりにする | §7 を見る |

経緯と実測は [../Old/docs/containers.md](../Old/docs/containers.md)。